# Pathology Hub Governed Cleanup v10.5.2 — API Proof Only

This notebook does **not** promote or modify live artifacts. It only proves `POST /evidence/search` with the working API key.

It prefers Colab secret `HUB_API`, then `PATHOLOGY_HUB_API_KEY`, then `X-API-Key`, then GCP Secret Manager `pathology-hub-api-key`. It sends the value under HTTP header `X-API-Key`.


In [ ]:

# Cell 1 — config/auth/helpers
import os, re, json, time, hashlib, subprocess, datetime, zipfile
from pathlib import Path
from datetime import timezone
import requests

PROJECT_ID = "pathology-annotation-project"
BASE_URL = "https://pathology-hub-v04-vorn5q2kga-uc.a.run.app"
RUN_ID = datetime.datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT = Path(f"/content/pathology_hub_api_proof_v10_5_2_{RUN_ID}/output")
OUT.mkdir(parents=True, exist_ok=True)
UPLOAD_TO_GCS = True
AUDIT_GCS = "gs://pathology_hub/06_audits/tags/governance/v10_5_2"

try:
    from google.colab import auth, userdata, files
    auth.authenticate_user()
    IN_COLAB = True
    print("Authenticated in Colab")
except Exception as e:
    IN_COLAB = False
    userdata = None
    files = None
    print("Not in Colab or auth skipped:", repr(e))

def now():
    return datetime.datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def sh(cmd, check=True, timeout=300):
    print(f"\n[{now()}] START: {cmd}", flush=True)
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)
    if p.stdout:
        print(p.stdout[-8000:], flush=True)
    print(f"[{now()}] DONE rc={p.returncode}", flush=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed rc={p.returncode}: {cmd}")
    return p

sh(f"gcloud config set project {PROJECT_ID}", check=False)

def clean_key(v):
    if not v:
        return None
    v = str(v).strip()
    v = re.sub(r"^X-API-Key\s*:\s*", "", v, flags=re.I).strip()
    v = re.sub(r"^Bearer\s+", "", v, flags=re.I).strip()
    v = v.strip("'\"")
    return v or None

def key_fingerprint(k):
    return {"loaded": bool(k), "length": len(k) if k else 0, "sha256_first8": hashlib.sha256(k.encode()).hexdigest()[:8] if k else None}


In [ ]:

# Cell 2 — load and test API key candidates
candidate_names = ["HUB_API", "PATHOLOGY_HUB_API_KEY", "X-API-Key"]
candidates = {}
if userdata:
    for name in candidate_names:
        try:
            k = clean_key(userdata.get(name))
            if k:
                candidates[f"COLAB:{name}"] = k
        except Exception as e:
            print(f"{name}: not accessible: {repr(e)}")

# Try GCP Secret Manager as a fallback. This is not Colab userdata; it is project Secret Manager.
p = sh("gcloud secrets versions access latest --secret=pathology-hub-api-key --project=pathology-annotation-project", check=False)
if p.returncode == 0:
    k = clean_key(p.stdout)
    if k:
        candidates["GCP_SECRET_MANAGER:pathology-hub-api-key"] = k
else:
    print("GCP Secret Manager fallback failed or unavailable.")

print("Candidate fingerprints:")
for name, k in candidates.items():
    print(name, key_fingerprint(k))

probe_payload = {"query":"prostate adenocarcinoma cribriform pattern 4", "sources":["textbooks","pathout"], "max_results":1, "compact":True, "excerpt_char_limit":500}
working_name = None
working_key = None
candidate_statuses = []
for name, k in candidates.items():
    r = requests.post(f"{BASE_URL}/evidence/search", headers={"X-API-Key": k}, json=probe_payload, timeout=60)
    candidate_statuses.append({"candidate": name, "fingerprint": key_fingerprint(k), "status_code": r.status_code, "text_excerpt": r.text[:500]})
    print(name, "status", r.status_code, "excerpt", r.text[:200])
    if r.status_code == 200 and working_key is None:
        working_name, working_key = name, k

if not working_key:
    raise RuntimeError("No working API key candidate found. Check that HUB_API contains the current Cloud Run API key and notebook access is enabled.")

API_KEY = working_key
print("WORKING API SECRET:", working_name, key_fingerprint(API_KEY))


In [ ]:

# Cell 3 — run governed cleanup API proof
FORBIDDEN_PATTERNS = [
    "::Lectures::", "::Textbooks::", "Slide_", "Page_", "Digital_Pathology_Slide", "Pathology_Slide", "Benign_Cystic_Neck_Mass_Case_01", "::Error"
]

def find_primary_tags(obj):
    tags=[]
    if isinstance(obj, dict):
        for k,v in obj.items():
            if k in {"primary_tag", "primary_tag_governed", "tag"} and isinstance(v, str):
                tags.append(v)
            else:
                tags.extend(find_primary_tags(v))
    elif isinstance(obj, list):
        for x in obj:
            tags.extend(find_primary_tags(x))
    return tags

def forbidden_tags_in_response(js):
    tags=find_primary_tags(js)
    bad=[]
    for t in tags:
        if t == "__UNMAPPED__" or any(p in t for p in FORBIDDEN_PATTERNS):
            bad.append(t)
    return sorted(set(bad))

health = requests.get(f"{BASE_URL}/health", timeout=60)
print("health", health.status_code, health.text[:500])

payloads = [
    {"query":"melanoma invasive overview", "sources":["lectures"], "max_results":5, "compact":True, "excerpt_char_limit":900},
    {"query":"ovarian high grade serous carcinoma p53 BRCA", "sources":["who","textbooks","pathout","journals"], "max_results":5, "compact":True, "excerpt_char_limit":900},
    {"query":"prostate adenocarcinoma cribriform pattern 4", "sources":["textbooks","pathout"], "max_results":5, "compact":True, "excerpt_char_limit":900},
]
search_tests=[]
for payload in payloads:
    r = requests.post(f"{BASE_URL}/evidence/search", headers={"X-API-Key": API_KEY}, json=payload, timeout=90)
    try:
        js = r.json()
    except Exception:
        js = None
    bad = forbidden_tags_in_response(js) if js is not None else []
    rec = {"payload":payload, "status_code":r.status_code, "text_excerpt":r.text[:1000], "json_keys": list(js.keys()) if isinstance(js,dict) else [], "forbidden_returned_primary_tags": bad, "forbidden_count": len(bad)}
    search_tests.append(rec)
    print("query", payload["query"], "status", r.status_code, "forbidden", len(bad))
    if r.status_code != 200:
        print(r.text[:500])

proof = {
    "schema_version":"pathology_hub_governed_cleanup_api_proof.v10_5_2",
    "generated_at_utc": now(),
    "working_secret_used": working_name,
    "candidate_statuses": candidate_statuses,
    "health": {"status_code": health.status_code, "text_excerpt": health.text[:5000]},
    "health_json": health.json() if health.headers.get("content-type","").startswith("application/json") or health.text.strip().startswith("{") else None,
    "search_tests": search_tests,
    "api_proof_passed": bool(health.status_code == 200 and all(t["status_code"] == 200 and t["forbidden_count"] == 0 for t in search_tests)),
}
proof_path = OUT / "PATHOLOGY_HUB_GOVERNED_CLEANUP_API_PROOF_v10_5_2.json"
proof_path.write_text(json.dumps(proof, indent=2), encoding="utf-8")
print(json.dumps({"api_proof_passed": proof["api_proof_passed"], "statuses":[t["status_code"] for t in search_tests], "forbidden_counts":[t["forbidden_count"] for t in search_tests], "working_secret_used": working_name}, indent=2))
if not proof["api_proof_passed"]:
    raise RuntimeError("API proof failed. See PATHOLOGY_HUB_GOVERNED_CLEANUP_API_PROOF_v10_5_2.json")


In [ ]:

# Cell 4 — upload proof and download output ZIP
summary = {
    "schema_version":"pathology_hub_governed_cleanup_api_proof_summary.v10_5_2",
    "generated_at_utc": now(),
    "api_proof_passed": proof["api_proof_passed"],
    "working_secret_used": working_name,
    "search_status_codes": [t["status_code"] for t in search_tests],
    "forbidden_counts": [t["forbidden_count"] for t in search_tests],
}
(OUT / "PATHOLOGY_HUB_GOVERNED_CLEANUP_API_PROOF_SUMMARY_v10_5_2.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
md = "# Pathology Hub Governed Cleanup v10.5.2 API Proof\n\n" + json.dumps(summary, indent=2)
(OUT / "PATHOLOGY_HUB_GOVERNED_CLEANUP_API_PROOF_SUMMARY_v10_5_2.md").write_text(md, encoding="utf-8")

if UPLOAD_TO_GCS:
    sh(f"gcloud storage cp {OUT}/PATHOLOGY_HUB_GOVERNED_CLEANUP_API_PROOF_v10_5_2.json {AUDIT_GCS}/PATHOLOGY_HUB_GOVERNED_CLEANUP_API_PROOF_v10_5_2.json", check=True)
    sh(f"gcloud storage cp {OUT}/PATHOLOGY_HUB_GOVERNED_CLEANUP_API_PROOF_SUMMARY_v10_5_2.json {AUDIT_GCS}/PATHOLOGY_HUB_GOVERNED_CLEANUP_API_PROOF_SUMMARY_v10_5_2.json", check=True)

zip_path = OUT.parent / "PATHOLOGY_HUB_GOVERNED_CLEANUP_API_PROOF_v10_5_2_OUTPUTS.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.glob("*"):
        z.write(p, arcname=f"output/{p.name}")
print("Output ZIP:", zip_path)
if files:
    files.download(str(zip_path))
